# CNN78u: validation-driven learning-rate schedule

Use the same fixed 20% training subset, original four-block CNN, Xavier uniform
initialization, dropout 0.5, batch size 64 and Adam weight decay 0. Start from fresh
weights with seed 69; do not continue either earlier checkpoint.

The experiment adds ReduceLROnPlateau and increases the maximum budget to 15
epochs. Because both schedule and budget change, comparison with the earlier
three-epoch run cannot isolate the scheduler's causal effect.

Upload **only this notebook** alongside the existing `cnn78u_colab_bundle.zip` in
`MyDrive/Colab Notebooks/EC/Project/Demo`. Choose a GPU runtime. The helper source
is embedded. Outputs go into a new `scheduler_runs` folder. There is no test
inference, residual model, or Kaiming initialization in this experiment.


## 1. Dependencies

Keep Colab's preinstalled CUDA-enabled PyTorch. Install only missing reporting/data packages. Python code and outputs are in English. The local-test branch is for repository QA and does not mount Drive or evaluate the real held-out test set.

In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
LOCAL_TEST = os.environ.get("CNN78U_LOCAL_NOTEBOOK_TEST") == "1"
packages = {"numpy":"numpy", "pandas":"pandas", "pyarrow":"pyarrow", "PIL":"Pillow",
            "sklearn":"scikit-learn", "matplotlib":"matplotlib", "tqdm":"tqdm", "psutil":"psutil"}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
if importlib.util.find_spec("torch") is None:
    raise RuntimeError("PyTorch is missing. Select a standard Colab GPU runtime.")

## 2. Mount Drive

Authentication is performed by Colab. No username, password, token, or service-account key is embedded in this notebook.

In [ ]:
if LOCAL_TEST:
    DRIVE_DEMO = Path(os.environ["CNN78U_LOCAL_DEMO"])
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DEMO = Path("/content/drive/MyDrive/Colab Notebooks/EC/Project/Demo")
assert DRIVE_DEMO.is_dir(), f"Upload files first: {DRIVE_DEMO}"
print("Drive project:", DRIVE_DEMO)

## 3. Stage data on the runtime disk and verify the upload

Training reads numeric memmaps from the VM's local disk, not thousands of small Drive reads. The preferred upload is the provided ZIP. An expanded bundle or project tree with `us_pipeline/` and the current prepared directory is also accepted. Copy/unpack/hash time is measured separately from training.

[Colab's official FAQ](https://research.google.com/colaboratory/faq.html) recommends avoiding many small Drive I/O operations. Runtime storage is temporary; checkpoints and reports are copied back to Drive.

In [ ]:
import hashlib, json, shutil, tempfile, time, zipfile
staging_start = time.perf_counter()
cache_parent = None if LOCAL_TEST else "/content"
LOCAL_CACHE = Path(tempfile.mkdtemp(prefix="cnn78u_", dir=cache_parent))
LOCAL_PROJECT = LOCAL_CACHE / "project"
LOCAL_PROJECT.mkdir()
archive_candidates = [DRIVE_DEMO / "cnn78u_colab_bundle.zip",
                      DRIVE_DEMO / "colab/cnn78u_colab_bundle.zip",
                      DRIVE_DEMO / "ImageTrading-CV-Project/colab/cnn78u_colab_bundle.zip"]
archive_source = next((p for p in archive_candidates if p.is_file()), None)
if archive_source:
    archive = LOCAL_CACHE / "bundle.zip"
    shutil.copy2(archive_source, archive)
    with zipfile.ZipFile(archive) as z:
        required = sum(info.file_size for info in z.infolist())
        if required > shutil.disk_usage(LOCAL_CACHE).free:
            raise RuntimeError("Not enough runtime disk space to unpack the bundle")
        for info in z.infolist():
            destination = (LOCAL_PROJECT / info.filename).resolve()
            if not destination.is_relative_to(LOCAL_PROJECT.resolve()):
                raise ValueError("Unsafe archive member path")
        z.extractall(LOCAL_PROJECT)
else:
    candidates = [DRIVE_DEMO, DRIVE_DEMO / "cnn78u_colab_bundle", DRIVE_DEMO / "ImageTrading-CV-Project"]
    source = next((p for p in candidates if (p / "us_pipeline/models.py").is_file()), None)
    if source is None:
        raise FileNotFoundError("Upload cnn78u_colab_bundle.zip next to this notebook")
    prepared_source = next((p for p in [source / "prepared", source / "artifacts/cnn78u_5min_h1/prepared"]
                            if (p / "manifest.json").is_file()), None)
    if prepared_source is None:
        raise FileNotFoundError("Missing prepared next-bar dataset; raw bars alone are insufficient here")
    shutil.copytree(source / "us_pipeline", LOCAL_PROJECT / "us_pipeline", ignore=shutil.ignore_patterns("__pycache__"))
    shutil.copytree(prepared_source, LOCAL_PROJECT / "prepared")
    if (source / "bundle_manifest.json").exists():
        shutil.copy2(source / "bundle_manifest.json", LOCAL_PROJECT / "bundle_manifest.json")
verified_upload = False
if (LOCAL_PROJECT / "bundle_manifest.json").exists():
    upload = json.loads((LOCAL_PROJECT / "bundle_manifest.json").read_text())
    for relative, details in upload["files"].items():
        path = (LOCAL_PROJECT / relative).resolve()
        assert path.is_relative_to(LOCAL_PROJECT.resolve()), relative
        assert hashlib.sha256(path.read_bytes()).hexdigest() == details["sha256"], relative
    verified_upload = True
sys.path.insert(0, str(LOCAL_PROJECT))
PREPARED = LOCAL_PROJECT / "prepared"
STAGING_SECONDS = time.perf_counter() - staging_start
print({"local_project": str(LOCAL_PROJECT), "staging_seconds": round(STAGING_SECONDS, 2),
       "upload_checksums_verified": verified_upload})

## 4. Install the embedded experiment helper
The source is staged locally and copied into the experiment folder for provenance.
Original notebooks, uploaded data, and previous experiment directories are preserved.

In [ ]:
import importlib
helper = LOCAL_PROJECT / "scheduler_runner.py"
helper.write_text('"""CNN78u validation-driven scheduler experiment; training/validation only."""\nimport hashlib\nimport json\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Subset\n\nfrom us_pipeline.dataset import ImageDataset\nfrom us_pipeline.models import build_model\nfrom us_pipeline.reporting import binary_metrics, write_evaluation\nfrom us_pipeline.experiment import (make_loader, predict, save_json, save_checkpoint,\n                                    mirror, sync_tree, sync_cuda, runtime_info)\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef fixed_indices(n, fraction=.2, seed=69):\n    if not 0 < fraction <= 1 or n < 1:\n        raise ValueError(\'Invalid sampling contract\')\n    count = max(1, int(np.floor(n * fraction)))\n    return np.sort(np.random.default_rng(seed).choice(n, count, replace=False))\n\n\ndef probability_summary(p):\n    p = np.asarray(p, dtype=np.float64)\n    return dict(probability_std=float(p.std()), probability_min=float(p.min()),\n                probability_max=float(p.max()), unique_probabilities=len(np.unique(p)),\n                predicted_up_fraction=float((p >= .5).mean()))\n\n\ndef collapse_signal(health):\n    # Both constant predictions AND near-zero feature parameters are required.\n    # This is a numerical-failure guard, not a significance test for weak alpha.\n    return (health[\'train\'][\'probability_std\'] <= 1e-8\n            and health[\'validation\'][\'probability_std\'] <= 1e-8\n            and health[\'feature_parameter_max_abs\'] <= 1e-8)\n\n\ndef seed_all(seed):\n    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef probe_tensor(ds, indices):\n    rows = [ds[int(i)] for i in indices]\n    return torch.stack([r[0] for r in rows]), torch.tensor([r[1] for r in rows])\n\n\ndef health_check(model, probes, device, gradients, step):\n    was_training = model.training\n    model.eval(); result = {\'step\': step, \'gradient_norms_last_training_batch\': gradients}\n    with torch.no_grad():\n        for name, (x, y) in probes.items():\n            p = torch.cat([model(batch.to(device)).softmax(1)[:, 1].cpu()\n                           for batch in x.split(16)]).numpy()\n            result[name] = {**probability_summary(p),\n                            \'accuracy\': float(((p >= .5) == y.numpy()).mean()),\n                            \'log_loss\': binary_metrics(y.numpy(), p)[\'log_loss\']}\n        weights = {name: float(p.detach().abs().max()) for name,p in model.blocks.named_parameters()\n                   if name.endswith(\'weight\')}\n        result[\'feature_parameter_max_abs\'] = max(weights.values())\n        result[\'feature_weight_max_abs\'] = weights\n        result[\'batchnorm_scale_max_abs\'] = {str(i):float(m.weight.abs().max())\n            for i,m in enumerate(model.blocks) if isinstance(m, nn.BatchNorm2d)}\n        z = probes[\'train\'][0][:16].to(device); activation = {}\n        for i, layer in enumerate(model.blocks):\n            z = layer(z)\n            if isinstance(layer, nn.ReLU):\n                activation[str(i)] = {\'zero_fraction\':float((z==0).float().mean()),\n                    \'between_sample_std\':float(z.std(dim=0, unbiased=False).mean())}\n        result[\'relu_activations\'] = activation\n    model.train(was_training)\n    result[\'collapse_signal\'] = collapse_signal(result)\n    return result\n\n\nclass ScheduleControl:\n    """Shared absolute improvement tolerance for scheduling and early stopping."""\n    def __init__(self, optimizer, settings):\n        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(\n            optimizer, mode=\'min\', factor=settings[\'lr_factor\'],\n            patience=settings[\'lr_patience\'], threshold=settings[\'min_delta\'],\n            threshold_mode=\'abs\', cooldown=0, min_lr=settings[\'min_lr\'])\n        self.best = float(\'inf\')\n        self.stale = 0\n        self.patience = settings[\'early_stop_patience\']\n        self.min_delta = settings[\'min_delta\']\n\n    def step(self, validation_loss):\n        if not np.isfinite(validation_loss):\n            raise ValueError(\'Nonfinite scheduler metric\')\n        if validation_loss < self.best - self.min_delta:\n            self.best = validation_loss; self.stale = 0\n        else:\n            self.stale += 1\n        self.scheduler.step(validation_loss)\n        return self.stale >= self.patience\n\n    def state_dict(self):\n        return {\'scheduler\':self.scheduler.state_dict(), \'best\':self.best, \'stale\':self.stale}\n\n    def load_state_dict(self, state):\n        self.scheduler.load_state_dict(state[\'scheduler\'])\n        self.best = state[\'best\']; self.stale = state[\'stale\']\n\n\ndef run_experiment(prepared, root, persistent, indices, probe_ids, settings, weight_decay, smoke=False):\n    """Epoch-boundary resume; terminal collapsed/completed runs are read-only cached."""\n    if weight_decay != 0:\n        raise ValueError(\'This protocol fixes weight decay at zero\')\n    root, persistent = Path(root), Path(persistent)\n    root.mkdir(parents=True, exist_ok=True); persistent.mkdir(parents=True, exist_ok=True)\n    train = ImageDataset(prepared, \'train\'); validation = ImageDataset(prepared, \'validation\')\n    if train.config[\'lookback\'] != 78 or train.config[\'horizon\'] != 1:\n        raise ValueError(\'Expected 78-bar input and next-bar target\')\n    config = train.config\n    contract = {\'settings\':settings,\'weight_decay\':weight_decay,\'config\':config,\'smoke\':smoke,\n                \'indices_sha256\':hashlib.sha256(indices.tobytes()).hexdigest(),\n                \'probes\':{k:v.tolist() for k,v in probe_ids.items()},\n                \'source_sha256\':sha(__file__),\n                \'data\':{n:sha(Path(prepared)/n) for n in [\'manifest.json\',\'features.npy\',\n                       \'multipliers.npy\',\'valid.npy\',\'train.parquet\',\'validation.parquet\']}}\n    signature = hashlib.sha256(json.dumps(contract,sort_keys=True).encode()).hexdigest()\n    if (root/\'contract.json\').exists():\n        if json.loads((root/\'contract.json\').read_text()) != contract:\n            raise ValueError(\'Run contract changed; use a new experiment ID\')\n    save_json(root/\'contract.json\', contract, persistent)\n    if (root/\'summary.json\').exists():\n        return json.loads((root/\'summary.json\').read_text())\n    seed_all(settings[\'seed\'])\n    model = build_model(config).to(device := torch.device(settings[\'device\']))\n    initial_sha = hashlib.sha256(b\'\'.join(p.detach().cpu().numpy().tobytes()\n                                         for p in model.parameters())).hexdigest()\n    optimizer = torch.optim.Adam(model.parameters(),lr=settings[\'learning_rate\'],weight_decay=weight_decay)\n    schedule = ScheduleControl(optimizer, settings)\n    loader_settings = {**settings, \'amp\':False}\n    train_loader = make_loader(Subset(train, indices.tolist()), loader_settings, True)\n    val_view = Subset(validation, probe_ids[\'validation\'].tolist()) if smoke else validation\n    val_loader = make_loader(val_view, loader_settings, False)\n    probes = {\'train\':probe_tensor(train,probe_ids[\'train\']),\n              \'validation\':probe_tensor(validation,probe_ids[\'validation\'])}\n    prior = float(train.samples.iloc[indices].label.mean())\n    history=[]; health=[]; best=float(\'inf\'); first=1; steps=0; strikes=0\n    early_stopped=False\n    if (root/\'last.pt\').exists():\n        ck=torch.load(root/\'last.pt\',map_location=\'cpu\',weights_only=False)\n        if ck[\'signature\'] != signature: raise ValueError(\'Resume signature mismatch\')\n        model.load_state_dict(ck[\'model\']); optimizer.load_state_dict(ck[\'optimizer\'])\n        schedule.load_state_dict(ck[\'schedule\'])\n        early_stopped=ck[\'early_stopped\']\n        history=ck[\'history\']; health=ck[\'health\']; best=ck[\'best\']; steps=ck[\'steps\']\n        strikes=ck[\'strikes\']; first=ck[\'epoch\']+1\n        torch.set_rng_state(ck[\'torch_rng\'])\n        if device.type==\'cuda\': torch.cuda.set_rng_state_all(ck[\'cuda_rng\'])\n    else:\n        health.append(health_check(model,probes,device,{},0))\n    save_json(root/\'health.json\',health,persistent)\n    stopped=False; stop_reason=None\n    for epoch in ([] if early_stopped else range(first,settings[\'epochs\']+1)):\n        lr_used=optimizer.param_groups[0][\'lr\']\n        model.train(); train_loader.generator.manual_seed(settings[\'seed\']+epoch)\n        if device.type==\'cuda\': torch.cuda.reset_peak_memory_stats(device)\n        sync_cuda(device); started=time.perf_counter(); total=correct=seen=0; monitor_seconds=0.\n        for batch_index,(x,y,_) in enumerate(train_loader):\n            x,y=x.to(device),y.to(device); optimizer.zero_grad(set_to_none=True)\n            logits=model(x); loss=nn.functional.cross_entropy(logits,y)\n            if not torch.isfinite(loss):\n                stopped=True; stop_reason=\'nonfinite_loss\'; break\n            loss.backward()\n            monitor=(steps+1)%settings[\'monitor_every\']==0 or batch_index+1==len(train_loader)\n            gradients={n:float(p.grad.detach().double().norm()) for n,p in model.named_parameters()\n                       if p.grad is not None} if monitor else {}\n            if monitor and not all(np.isfinite(list(gradients.values()))):\n                stopped=True; stop_reason=\'nonfinite_gradient\'; break\n            optimizer.step(); steps+=1\n            seen+=len(y); total+=float(loss.detach())*len(y); correct+=int((logits.argmax(1)==y).sum())\n            if monitor:\n                sync_cuda(device); t=time.perf_counter()\n                h=health_check(model,probes,device,gradients,steps); h[\'epoch\']=epoch\n                health.append(h); strikes=strikes+1 if h[\'collapse_signal\'] else 0\n                save_json(root/\'health.json\',health,persistent)\n                sync_cuda(device); monitor_seconds+=time.perf_counter()-t\n                print(json.dumps({\'epoch\':epoch,\'step\':steps,\'train_loss\':total/seen,\n                    \'probe_probability_std\':h[\'validation\'][\'probability_std\'],\n                    \'feature_max_abs\':h[\'feature_parameter_max_abs\'],\'collapse_strikes\':strikes}),flush=True)\n                if strikes>=3:\n                    stopped=True; stop_reason=\'three_consecutive_feature_collapse_checks\'; break\n        sync_cuda(device); train_wall=time.perf_counter()-started\n        frame,timing=predict(model,validation,val_loader,device,False,not smoke)\n        if not np.isfinite(frame.probability_up).all():\n            stopped=True; stop_reason=\'nonfinite_validation_probability\'\n            metrics={}\n        else:\n            metrics=binary_metrics(frame.label,frame.probability_up)\n        if not stopped:\n            early_stopped=schedule.step(metrics[\'log_loss\'])\n        row={\'epoch\':epoch,\'learning_rate_used\':lr_used,\n             \'learning_rate_next\':optimizer.param_groups[0][\'lr\'],\n             \'early_stop_bad_epochs\':schedule.stale,\n             \'train_probe_eval_loss\':health[-1][\'train\'][\'log_loss\'],\n             \'train_probe_eval_accuracy\':health[-1][\'train\'][\'accuracy\'],\'steps_total\':steps,\'samples_seen\':seen,\'partial_epoch\':stopped,\n             \'train_loss\':total/seen if seen else None,\'train_accuracy\':correct/seen if seen else None,\n             \'train_wall_seconds_including_monitor\':train_wall,\'monitor_seconds\':monitor_seconds,\n             \'validation_seconds\':timing[\'end_to_end_seconds\'],\n             \'train_samples_per_second_including_monitor\':seen/train_wall,\n             \'gpu_peak_allocated_bytes\':torch.cuda.max_memory_allocated(device) if device.type==\'cuda\' else None,\n             **{\'val_\'+k:v for k,v in metrics.items() if not isinstance(v,list)}}\n        history.append(row)\n        ck={\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'config\':config,\n            \'schedule\':schedule.state_dict(),\'early_stopped\':early_stopped,\n            \'signature\':signature,\'history\':history,\'health\':health,\'steps\':steps,\'epoch\':epoch,\n            \'strikes\':strikes,\'torch_rng\':torch.get_rng_state(),\n            \'cuda_rng\':torch.cuda.get_rng_state_all() if device.type==\'cuda\' else None}\n        if not stopped and metrics[\'log_loss\']<best:\n            best=metrics[\'log_loss\']; ck[\'best\']=best\n            save_checkpoint(root/\'best.pt\',ck,persistent)\n            frame.to_parquet(root/\'best_validation_predictions.parquet\',index=False)\n            mirror(root/\'best_validation_predictions.parquet\',persistent)\n        ck[\'best\']=best\n        save_checkpoint(root/(\'stopped.pt\' if stopped else \'last.pt\'),ck,persistent)\n        save_json(root/\'history.json\',history,persistent)\n        save_json(root/\'health.json\',health,persistent)\n        print(json.dumps(row),flush=True)\n        if stopped or early_stopped: break\n    # A best pre-collapse checkpoint remains explicitly distinguished from terminal health.\n    evaluation=None\n    if (root/\'best_validation_predictions.parquet\').exists():\n        frame=pd.read_parquet(root/\'best_validation_predictions.parquet\')\n        evaluation=write_evaluation(frame,root/\'validation\',prior,\'validation\',{},\n            seed=settings[\'seed\'],bootstrap_repetitions=100 if smoke else 2000,smoke_only=smoke)\n        sync_tree(root/\'validation\',persistent/\'validation\')\n    result={\'status\':\'stopped_unhealthy\' if stopped else (\'early_stopped\' if early_stopped else \'completed_budget\'),\n            \'schedule\':schedule.state_dict(),\n            \'best_epoch\':torch.load(root/\'best.pt\',map_location=\'cpu\',weights_only=False)[\'epoch\'] if (root/\'best.pt\').exists() else None,\n            \'stop_reason\':stop_reason or (\'validation_plateau\' if early_stopped else None),\'smoke_only\':smoke,\'test_evaluated\':False,\n            \'weight_decay\':weight_decay,\'initial_parameters_sha256\':initial_sha,\n            \'train_subset_sha256\':contract[\'indices_sha256\'],\'training_samples\':len(indices),\n            \'training_prior\':prior,\'steps\':steps,\'history\':history,\n            \'final_health\':health[-1],\'runtime\':runtime_info(device),\n            \'best_validation_metrics\':evaluation[\'metrics\'] if evaluation else None,\n            \'interpretation\':\'Diagnostic only. Completing the budget is not proof of generalization.\'}\n    save_json(root/\'summary.json\',result,persistent)\n    return result\n')
importlib.invalidate_caches()
import scheduler_runner as runner

## 5. Freeze the training protocol
- Initial LR 0.001; factor 0.5; scheduler patience 2; minimum LR 0.00001.
- Both plateau detection and early stopping use an absolute improvement tolerance
  of 0.00001 in full-validation log loss. A new best checkpoint uses the actual
  lowest loss, even if its change is below this tolerance.
- **PyTorch patience semantics:** patience=2 permits two bad epochs and reduces
  LR after the third consecutive non-improving epoch. The new LR applies to the
  following epoch. It does not reduce LR every two epochs.
- Stop after six consecutive non-improving epochs, or at 15 epochs, whichever
  comes first. This allows training after the first LR reduction; a second
  reduction may coincide with early stopping and therefore never be used.
- Numerical-collapse monitoring remains active every 500 updates and at epoch end.

All epoch metrics use the complete validation split. Fixed train and validation
probes use eval mode, so dropout is disabled and BatchNorm running statistics are
not changed. Probe loss/accuracy are diagnostics, not replacement validation scores.

In [ ]:
import numpy as np, pandas as pd, torch
from datetime import datetime, timezone
from IPython.display import display, Image
SMOKE_MODE = LOCAL_TEST
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda" and not SMOKE_MODE:
    raise RuntimeError("Select a GPU runtime")
torch.set_num_threads(min(4, os.cpu_count() or 1))
if DEVICE == "cuda":
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
SETTINGS = dict(epochs=2 if SMOKE_MODE else 15,
                batch_size=4 if SMOKE_MODE else 64,
                num_workers=0 if SMOKE_MODE else 2,
                learning_rate=0.001, seed=69,
                monitor_every=1 if SMOKE_MODE else 500, device=DEVICE,
                lr_factor=0.5, lr_patience=2, min_lr=0.00001,
                min_delta=0.00001, early_stop_patience=6)
SAMPLING_RATE = 0.2
SAMPLE_SEED = 69
RESUME = False
EXPERIMENT_ID = ("scheduler_local_qa" if LOCAL_TEST else
                "cnn78u_scheduler_20pct_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc"))
PERSISTENT = DRIVE_DEMO / "scheduler_runs" / EXPERIMENT_ID
LOCAL_RUN = LOCAL_CACHE / "scheduler_run"
if RESUME:
    if not (PERSISTENT / "sampling.json").exists():
        raise FileNotFoundError("No experiment found for this ID")
    shutil.copytree(PERSISTENT, LOCAL_RUN)
else:
    if PERSISTENT.exists():
        raise FileExistsError("Use a new ID, or explicitly resume the existing experiment")
    PERSISTENT.mkdir(parents=True)
    LOCAL_RUN.mkdir()
display(pd.Series(SETTINGS, name="Frozen protocol"))
print("Results:", PERSISTENT)
print(runner.runtime_info(torch.device(DEVICE)))

## 6. Restore the same fixed 20% sampling contract
The random subset is identical to the previous repair notebook: seed 69, uniform
sampling without replacement, sorted original training-row indices. There is no
class balancing and no re-sampling between epochs. Full validation is retained.
The normal dataset yields 344,165 training examples and 5,378 updates per epoch;
15 complete epochs would give 80,670 updates. This is a budget cap, not a target
that overrides early stopping.

To resume after a disconnect, set RESUME=True and reuse the original ID with all
settings unchanged, then rerun from the beginning. The helper restores model,
optimizer, scheduler, early-stop counters, RNG and completed-epoch history.
A partly finished epoch is repeated. Completed runs return cached results. A
stopped unhealthy run is terminal once its summary is saved.

In [ ]:
training = runner.ImageDataset(PREPARED, "train")
validation = runner.ImageDataset(PREPARED, "validation")
assert training.config["model"]["channels"] == [32,64,128,256]
assert training.config["model"]["dropout"] == 0.5
indices = runner.fixed_indices(len(training), SAMPLING_RATE, SAMPLE_SEED)
if SMOKE_MODE:
    indices = indices[:8]
probe_rng = np.random.default_rng(SAMPLE_SEED + 1)
probe_ids = {"train":np.sort(probe_rng.choice(indices, min(128,len(indices)),replace=False)),
             "validation":np.sort(probe_rng.choice(len(validation),8 if SMOKE_MODE else min(128,len(validation)),replace=False))}
sampling = dict(sampling_rate=SAMPLING_RATE, sampling_seed=SAMPLE_SEED,
    count=len(indices), eligible_count=len(training), smoke_only=SMOKE_MODE,
    indices_sha256=hashlib.sha256(indices.tobytes()).hexdigest(),
    train_index_file_sha256=runner.sha(PREPARED / "train.parquet"))
if RESUME:
    assert json.loads((LOCAL_RUN / "sampling.json").read_text()) == sampling
    assert np.array_equal(np.load(LOCAL_RUN / "train_indices.npy"),indices)
else:
    np.save(LOCAL_RUN / "train_indices.npy",indices)
    runner.mirror(LOCAL_RUN / "train_indices.npy",PERSISTENT)
    np.savez(LOCAL_RUN / "probe_indices.npz",**probe_ids)
    runner.mirror(LOCAL_RUN / "probe_indices.npz",PERSISTENT)
    runner.save_json(LOCAL_RUN / "sampling.json",sampling,PERSISTENT)
    shutil.copy2(helper,LOCAL_RUN / "scheduler_runner.py")
    runner.mirror(LOCAL_RUN / "scheduler_runner.py",PERSISTENT)
# If prior repair indices are present, verify exact equality for the same data contract.
matched_previous = []
if not SMOKE_MODE:
    for previous in (DRIVE_DEMO / "repair_runs").glob("*/sampling.json"):
        previous_contract = json.loads(previous.read_text())
        if (not previous_contract.get("smoke_only") and
            previous_contract.get("train_index_file_sha256") == sampling["train_index_file_sha256"] and
            previous_contract.get("requested_sampling_rate") == SAMPLING_RATE and
            previous_contract.get("sampling_seed") == SAMPLE_SEED):
            assert np.array_equal(np.load(previous.parent / "train_indices.npy"),indices)
            matched_previous.append(previous.parent.name)
display(pd.Series(sampling))
print("Matching prior runs:",matched_previous)
print("Retained training up fraction:",training.samples.iloc[indices].label.mean())
print("Full validation samples:",len(validation))

## 7. Train, validate, schedule and checkpoint
Order: train epoch -> full validation -> update scheduler/early-stop state -> save
checkpoint -> next epoch. History records both LR used and LR scheduled next.
Best checkpoint selection uses validation log loss only.

The original conservative collapse guard requires both near-constant probe
probabilities and near-zero convolution/BatchNorm scales for three consecutive
checks. Not triggering it is not proof of predictive ability. Nonfinite loss or
monitored gradients also stop training.

Checkpoints and reports are mirrored to Drive. Full validation happens every
epoch, so this can take several hours. The final budget may be smaller due to
early stopping. Do not infer a runtime guarantee from the earlier three-epoch run.

In [ ]:
result = runner.run_experiment(PREPARED,LOCAL_RUN / "training",PERSISTENT / "training",
                               indices,probe_ids,SETTINGS,weight_decay=0.0,smoke=SMOKE_MODE)
history = pd.DataFrame(result["history"])
display(history)
print("Status:",result["status"],"Reason:",result["stop_reason"])
print("Best validation epoch:",result["best_epoch"])
print("Test evaluated:",result["test_evaluated"])
history.to_csv(LOCAL_RUN / "history.csv",index=False)
runner.mirror(LOCAL_RUN / "history.csv",PERSISTENT)

## 8. Inspect learning rate, fit and numerical health
The fixed training-probe eval loss is directly comparable in mode to validation
loss, but its small sample size makes it noisy. The in-training loss additionally
contains dropout and changing weights. Inspect all three instead of treating
in-training and eval-mode accuracy as identical measurements.

In the second panel, both curves are fixed probes; full-validation loss is shown
separately. If training-probe fit improves but validation does not, investigate
generalization before increasing model complexity.

In [ ]:
import matplotlib.pyplot as plt
health=json.loads((LOCAL_RUN / "training/health.json").read_text())
fig,axes=plt.subplots(2,2,figsize=(12,8))
axes[0,0].step(history.epoch,history.learning_rate_used,where="mid",label="LR used")
axes[0,0].set_yscale("log");axes[0,0].set_title("Learning rate used per epoch")
axes[0,0].set_xlabel("Epoch");axes[0,0].legend()
for split in ["train","validation"]:
    axes[0,1].plot([h["step"] for h in health],[h[split]["log_loss"] for h in health],label=split+" probe (eval)")
axes[0,1].set_title("Fixed-probe log loss");axes[0,1].set_xlabel("Updates");axes[0,1].legend()
axes[1,0].plot(history.epoch,history.reindex(columns=["val_log_loss"])["val_log_loss"],marker="o",label="Full validation")
axes[1,0].set_title("Validation log loss");axes[1,0].set_xlabel("Epoch");axes[1,0].legend()
for split in ["train","validation"]:
    axes[1,1].plot([h["step"] for h in health],[max(h[split]["probability_std"],1e-12) for h in health],label=split)
axes[1,1].set_yscale("log");axes[1,1].set_title("Probe probability standard deviation")
axes[1,1].set_xlabel("Updates");axes[1,1].legend()
if SMOKE_MODE:
    fig.suptitle("SMOKE CHECK ONLY: not a performance estimate")
fig.tight_layout()
fig.savefig(LOCAL_RUN / "scheduler_diagnostics.png",dpi=140)
plt.close(fig)
runner.mirror(LOCAL_RUN / "scheduler_diagnostics.png",PERSISTENT)
display(Image(filename=str(LOCAL_RUN / "scheduler_diagnostics.png")))

## 9. Best-checkpoint validation and export
The best completed epoch may differ from terminal health. Compare classification,
ranking and probability metrics with the training-prior baseline. A majority-class
prediction can exceed 50% accuracy without discriminating examples. A tiny gain
on this repeatedly used validation period does not establish out-of-sample alpha.

No Kaiming or architecture change is made here. Once this training protocol is
assessed, a separate initialization comparison can hold it fixed. A fresh untouched
time period is required for a final independent test.

In [ ]:
report = LOCAL_RUN / "training/validation"
if result["best_validation_metrics"] is not None:
    display(pd.Series({k:v for k,v in result["best_validation_metrics"].items() if not isinstance(v,list)},name="Best validation"))
    display(pd.read_csv(report / "baselines.csv"))
    display(Image(filename=str(report / "evaluation.png")))
else:
    print("No eligible completed checkpoint. Inspect the stop reason and health records.")
summary = dict(experiment_id=EXPERIMENT_ID, status=result["status"],
    best_epoch=result["best_epoch"], completed_or_partial_epochs=len(history),
    optimizer_updates=result["steps"], smoke_only=SMOKE_MODE,test_evaluated=False,
    train_wall_seconds=float(history.train_wall_seconds_including_monitor.sum()),
    validation_seconds=float(history.validation_seconds.sum()),
    initialization="Original Xavier uniform",weight_decay=0,dropout=0.5)
runner.save_json(LOCAL_RUN / "experiment_summary.json",summary,PERSISTENT)
archive=Path(shutil.make_archive(str(LOCAL_CACHE / "scheduler_results"),"zip",root_dir=LOCAL_RUN))
shutil.copy2(archive,PERSISTENT / "scheduler_results.zip")
display(pd.Series(summary))
print("Saved:",PERSISTENT)